In [1]:
import subprocess
import os
import json
from pathlib import Path
import pdfplumber
import shutil
from pathlib import Path
import re

In [2]:
#%conda install -c conda-forge psycopg2

In [3]:
import psycopg2
from psycopg2 import sql, extras
import logging


In [4]:
# Set up basic logging to see success/failure messages
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

In [5]:
DB_PARAMS = {
    'dbname': 'thecall',
    'user': 'postgres',        
    'password': 'password',    
    'host': 'localhost',           
    'port': '5432'                 
}

In [6]:
def connect_to_db(params):
    """Establishes a connection to the PostgreSQL database."""
    conn = None
    try:
        logging.info("Attempting to connect to PostgreSQL database...")
        conn = psycopg2.connect(**params)
        # Set autocommit to False so we can control transactions (needed for UPDATE)
        conn.autocommit = False 
        logging.info("Connection successful.")
        return conn
    except psycopg2.Error as e:
        logging.error(f"Error connecting to the database: {e}")
        return None

In [7]:
def collect_data_from_db(conn, table_schema, table_name):
    try:
        with conn.cursor() as cur:
            # Correctly use sql.Identifier for both schema and table separately
            query = sql.SQL("SELECT id, filename, folder FROM {}.{} WHERE json_raw IS NULL LIMIT 10").format(
                sql.Identifier(table_schema),
                sql.Identifier(table_name)
            )
            cur.execute(query)
            return cur.fetchall()
    except psycopg2.Error as e:
        logging.error(f"Error fetching data: {e}")
        return []

In [8]:
def clean_archival_text(raw_text):
    if not raw_text: return ""
    # Remove the specific artifacts identified in previous runs
    text = raw_text.replace('ä½®©', '')
    # Strip non-printable characters and collapse extra spaces
    text = re.sub(r'[^\x20-\x7E\n\r\t]', '', text)
    text = re.sub(r' +', ' ', text)
    return text.strip()

In [9]:
def extract_to_standard_json(all_data, output_root, soffice_path, assembled_root):
    output_root = Path(output_root).resolve()
    assembled_root = Path(assembled_root).resolve()
    pdf_archive_dir = output_root / "archived_pdfs"
    pdf_archive_dir.mkdir(parents=True, exist_ok=True)
    
    # # CRITICAL: Create a clean environment dictionary
    # # This prevents LibreOffice from trying to use your Jupyter/Conda Python libraries
    # clean_env = {k: v for k, v in os.environ.items() if "PYTHON" not in k.upper()}
    # # Ensure SystemRoot is present for Windows DLL loading
    # if "SystemRoot" not in clean_env:
    #     clean_env["SystemRoot"] = os.environ.get("SystemRoot", "C:\\Windows")

    CONDA_VARS = {"CONDA_DEFAULT_ENV", "CONDA_EXE", "CONDA_PREFIX", "CONDA_PYTHON_EXE",
                  "CONDA_SHLVL", "CONDA_PROMPT_MODIFIER", "PYTHONPATH", "PYTHONHOME",
                  "PYTHONSTARTUP", "PYTHONUSERBASE"}
    
    clean_env = {k: v for k, v in os.environ.items() if k not in CONDA_VARS}

    
    update_payload = []

    for db_id, filename, folder in all_data:
        # Process only files without extensions (Quark files)
        if Path(filename).suffix != "":
            continue 

        source_file = assembled_root / folder / filename
        if not source_file.exists():
            print(f"!! File not found: {source_file}")
            continue

        print(f"Processing ID {db_id}: {filename}")
        
        try:
            # Using .as_uri() handles spaces and special characters (file:///E:/...)
            source_uri = source_file.as_uri()
            
            subprocess.run([
                soffice_path, "--headless", 
                "--convert-to", "pdf:draw_pdf_Export",
                "--outdir", str(pdf_archive_dir),
                source_uri
            ], check=True, env=clean_env, capture_output=True)
            
            # LibreOffice output name usually matches the source filename
            original_pdf = pdf_archive_dir / f"{filename}.pdf"
            id_based_pdf = pdf_archive_dir / f"{db_id}.pdf"
            
            if original_pdf.exists():
                if id_based_pdf.exists(): id_based_pdf.unlink()
                shutil.move(original_pdf, id_based_pdf)
            else:
                print(f"   !! PDF file not found after conversion for {filename}")
                continue

            # Standardized Extraction
            with pdfplumber.open(id_based_pdf) as pdf:
                json_data = {
                    "source_file": f"{folder}/{filename}".replace("\\", "/"),
                    "document_name": filename,
                    "page_count": len(pdf.pages),
                    "metadata": {"source_id": db_id, "format": "QuarkXPress Legacy"},
                    "pages": []
                }

                for i, page in enumerate(pdf.pages):
                    raw_content = page.extract_text() or ""
                    clean_content = clean_archival_text(raw_content)
                    json_data["pages"].append({
                        "page_name": str(i + 1),
                        "frames": [{
                            "bounds": [0, 0, 0, 0],
                            "paragraphs": [{"style": "Normal", "text": line} for line in clean_content.split('\n') if line.strip()],
                            "text": clean_content
                        }]
                    })
            
            update_payload.append((db_id, json.dumps(json_data, ensure_ascii=False)))
            
        except subprocess.CalledProcessError as e:
            error_msg = e.stderr.decode().strip() or e.stdout.decode().strip()
            print(f"   !! LibreOffice Error on ID {db_id}: {error_msg}")
        except Exception as e:
            print(f"   Failed processing ID {db_id}: {e}")

    return update_payload

In [10]:
def update_dbs(conn, table_schema, table_name, results):
    """
    Updates the database with extracted JSON content using the unique ID.
    
    :param conn: psycopg2 connection object
    :param table_schema: Schema name (e.g., 'articles')
    :param table_name: Table name (e.g., 'filelist')
    :param results: List of tuples [(db_id, json_string), ...]
    """
    if not results:
        logging.info("No results to update.")
        return

    # Use Identifier to safely handle schema and table names
    full_table_path = sql.Identifier(table_schema, table_name)

    try:
        with conn.cursor() as cur:
            # We use the unique ID for the join, which is much more efficient
            # t.id matches the first element of our tuple, t.json_content matches the second
            update_query = sql.SQL("""
                UPDATE {} AS f
                SET json_raw = t.json_content::jsonb
                FROM (VALUES %s) AS t(id, json_content)
                WHERE f.id = t.id::int
            """).format(full_table_path)
            
            # execute_values is highly efficient for batch updates
            extras.execute_values(cur, update_query, results)
            
            conn.commit()
            logging.info(f"Successfully updated {len(results)} rows in {table_schema}.{table_name}.")
            
    except psycopg2.Error as e:
        conn.rollback()
        logging.error(f"Database update failed: {e}")

In [11]:
def main():
    conn = connect_to_db(DB_PARAMS)

    INPUT_FOLDER = r"E:/Callproject/assembled"
    OUTPUT_FOLDER = r"E:/Callproject/c_out_quark"
    LIBREOFFICE_BIN = r"C:\Program Files\LibreOffice\program\soffice.exe" 
 
    if conn:
        try:
            table_schema = 'articles'
            table_name = 'filelist'

            # 1. Fetch existing entries
            all_data = collect_data_from_db(conn, table_schema, table_name)
            
            if not all_data:
                logging.warning("No data retrieved from DB. Exiting.")
                return

        # 2. Process files and generate JSON strings
            # Ensure ASSEMBLED_ROOT points to your 'E:\Callproject\assembled' base
            results = extract_to_standard_json(
                all_data, 
                OUTPUT_FOLDER, 
                LIBREOFFICE_BIN, 
                INPUT_FOLDER,
            )
            
            # 3. Batch update the DB
            update_dbs(conn, table_schema, table_name, results)
            
        finally:
            if conn:
                conn.close()
                logging.info("Database connection closed.")

In [12]:
if __name__ == "__main__":
    main()


INFO: Attempting to connect to PostgreSQL database...
INFO: Connection successful.


Processing ID 4: BLUFORD PIX PAGE 13
   !! LibreOffice Error on ID 4: Could not find platform independent libraries <prefix>
Error: source file could not be loaded
Processing ID 6: ENTERTAINMENT 11
   !! LibreOffice Error on ID 6: Error: source file could not be loaded
Processing ID 7: FRONT PAGE JUNE 20
   !! LibreOffice Error on ID 7: Could not find platform independent libraries <prefix>
Error: source file could not be loaded
Processing ID 8: Missouri Call Ads
   !! LibreOffice Error on ID 8: Could not find platform independent libraries <prefix>
Error: source file could not be loaded
Processing ID 9: NEWS PAGE 2 3  5 JUNE 20
   !! LibreOffice Error on ID 9: Could not find platform independent libraries <prefix>
Error: source file could not be loaded
Processing ID 10: NEWS PAGE 27
   !! LibreOffice Error on ID 10: Could not find platform independent libraries <prefix>
Error: source file could not be loaded
Processing ID 11: Page 14
   !! LibreOffice Error on ID 11: Error: source fil

INFO: No results to update.
INFO: Database connection closed.


   !! LibreOffice Error on ID 12: Error: source file could not be loaded
